In [2]:
import pandas as pd
import numpy as np
import os
import json

### 1. 生成sample_condition.tsv表

In [3]:
# 读取当前目录下的sample.tsv 
df = pd.read_csv('sample.tsv', sep='\t')
df['Time'] = df['Time'].fillna('')
df['Condition'] = df['Strain'] + ' ' + df['Carbon Sources'] + ' ' + df['Temperature'].fillna('NA') + ' ' + df['Time'].fillna('NA')
df

,Sample ID,Strain,Carbon Sources,Temperature,Time,Title,Series Accession,Related Sample,Condition
0,mt001001,wt,Barley,34°,,Comparative genomic analysis of the thermophil...,GSE27323,GSM675519,wt Barley 34°
1,mt001002,wt,Barley,45°,,Comparative genomic analysis of the thermophil...,GSE27323,"GSM675520,GSM675523",wt Barley 45°
2,mt001003,wt,Glucose,34°,,Comparative genomic analysis of the thermophil...,GSE27323,"GSM675524,GSM675525",wt Glucose 34°
3,mt001004,wt,Glucose,45°,,Comparative genomic analysis of the thermophil...,GSE27323,GSM675521,wt Glucose 45°
4,mt001005,wt,Alfalfa,45°,,Comparative genomic analysis of the thermophil...,GSE27323,GSM675522,wt Alfalfa 45°
...,...,...,...,...,...,...,...,...,...
103,mt014008,MtLat-1,Arabinan,45°,4d,The arabinose transporter MtLat-1 is involved ...,GSE221464,"GSM6873690,GSM6873691",MtLat-1 Arabinan 45° 4d
104,mt014009,MtAra,Aarabinose,45°,4h,The arabinose transporter MtLat-1 is involved ...,GSE221464,"GSM6873692,GSM6873693",MtAra Aarabinose 45° 4h
105,mt014010,MtAra,Xylose,45°,4h,The arabinose transporter MtLat-1 is involved ...,GSE221464,"GSM6873694,GSM6873695",MtAra Xylose 45° 4h
106,mt014011,MtAra,Arabinan,45°,4h,The arabinose transporter MtLat-1 is involved ...,GSE221464,"GSM6873696,GSM6873697",MtAra Arabinan 45° 4h


In [4]:
# df保留Sample Number、Sample ID、Series ID、Study Title、Species、Strain、Sample Name、Canbon Sources、Temperature列
df_1 = df[['Sample ID', 'Series Accession','Title', 'Condition']]

# 如果Conditon最后一位是_，则去掉
df_1.loc[:,'Condition'] = df_1['Condition'].apply(lambda x: x[:-1] if x[-1] == '_' else x)

df_1

,Sample ID,Series Accession,Title,Condition
0,mt001001,GSE27323,Comparative genomic analysis of the thermophil...,wt Barley 34°
1,mt001002,GSE27323,Comparative genomic analysis of the thermophil...,wt Barley 45°
2,mt001003,GSE27323,Comparative genomic analysis of the thermophil...,wt Glucose 34°
3,mt001004,GSE27323,Comparative genomic analysis of the thermophil...,wt Glucose 45°
4,mt001005,GSE27323,Comparative genomic analysis of the thermophil...,wt Alfalfa 45°
...,...,...,...,...
103,mt014008,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtLat-1 Arabinan 45° 4d
104,mt014009,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra Aarabinose 45° 4h
105,mt014010,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra Xylose 45° 4h
106,mt014011,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra Arabinan 45° 4h


In [5]:
# 读取dataset.tsv
df_2 = pd.read_csv('dataset.tsv', sep='\t')
df_2 = df_2[['Series Accession', 'PMID']]

df_2.head(20)

,Series Accession,PMID
0,GSE213575,37013645.0
1,GSE213573,37013645.0
2,GSE213570,37013645.0
3,GSE221464,36966330.0
4,GSE214002,36691040.0
5,GSE214142,36478865.0
6,GSE205182,36165620.0
7,GSE184074,35257374.0
8,GSE183387,NaN
9,GSE114057,31078793.0


In [6]:
# 将df_1和df_2合并，根据Series ID列
df_3 = pd.merge(df_1, df_2, on='Series Accession', how='left')

# 重置索引
df_3 = df_3.reset_index(drop=True)

df_3['PMID'] = df_3['PMID'].astype(str)
df_3 = df_3[df_3['PMID'] != 'nan']

# PMID去掉最后两个字符
df_3['PMID'] = df_3['PMID'].apply(lambda x: x[:-2] if x[-2:] == '.0' else x)

# 保存df_3为tsv文件
df_3.to_csv('sample_condition.tsv', sep='\t', index=False)

df_3

,Sample ID,Series Accession,Title,Condition,PMID
0,mt001001,GSE27323,Comparative genomic analysis of the thermophil...,wt Barley 34°,21964414
1,mt001002,GSE27323,Comparative genomic analysis of the thermophil...,wt Barley 45°,21964414
2,mt001003,GSE27323,Comparative genomic analysis of the thermophil...,wt Glucose 34°,21964414
3,mt001004,GSE27323,Comparative genomic analysis of the thermophil...,wt Glucose 45°,21964414
4,mt001005,GSE27323,Comparative genomic analysis of the thermophil...,wt Alfalfa 45°,21964414
...,...,...,...,...,...
103,mt014008,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtLat-1 Arabinan 45° 4d,36966330
104,mt014009,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra Aarabinose 45° 4h,36966330
105,mt014010,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra Xylose 45° 4h,36966330
106,mt014011,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra Arabinan 45° 4h,36966330


### 2. 根据基因id获取按文献分类的Exp

In [10]:
def get_gene_exp(gene):
    """
    根据基因id，返回该基因在各个文献对应样本的TPM表达量
    """

    df_exp_fname = pd.read_csv('exp.tsv', sep='\t')
    df_condition = pd.read_csv('sample_condition.tsv', sep='\t')

    # 根据gene id从df_exp_fname中取出对应的行
    df = df_exp_fname[df_exp_fname['Gene id'] == gene].copy()
    # df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)

    # 根据基因ID从exp表中，取出该基因在各个样本的表达量
    df_condition['Sample Exp'] = df_condition['Sample ID'].map(df.set_index('Gene id').iloc[:, 0:].T.to_dict()[gene])

    # df_condition返回的是一个dataframe；构造字典
    output_dict = {
        # 'gene_info': {
        'gene_id': {
            'gene_id': gene
        }
    }
    for _, row in df_condition.iterrows():
        pmid = str(row['PMID'])   # 原先是Publications，改叫为PMID
        if pmid not in output_dict['gene_id']:
            output_dict['gene_id'][pmid] = {
                'PMID': pmid,
                'Title': row['Title'],
            }
        sample_number = row['Sample ID']
        output_dict['gene_id'][pmid][sample_number] = {
            'Sample ID': sample_number,
            'Condition': row['Condition'],
            # 'Exp': str(row['Sample Exp'])
            'Exp': row['Sample Exp']
        }
    
    # # output_dict转为json格式
    # output_json = json.dumps(output_dict, indent=4)
    
    return output_dict

dict_gene_exp = get_gene_exp('MYCTH_101575')
dict_gene_exp

{'gene_id': {'gene_id': 'MYCTH_101575',
  '21964414': {'PMID': '21964414',
   'Title': 'Comparative genomic analysis of the thermophilic biomass-degrading fungi Myceliophthora thermophila and Thielavia terrestris.',
   'mt001001': {'Sample ID': 'mt001001',
    'Condition': 'wt Barley 34° ',
    'Exp': 13.59},
   'mt001002': {'Sample ID': 'mt001002',
    'Condition': 'wt Barley 45° ',
    'Exp': 4.45},
   'mt001003': {'Sample ID': 'mt001003',
    'Condition': 'wt Glucose 34° ',
    'Exp': 10.71},
   'mt001004': {'Sample ID': 'mt001004',
    'Condition': 'wt Glucose 45° ',
    'Exp': 3.92},
   'mt001005': {'Sample ID': 'mt001005',
    'Condition': 'wt Alfalfa 45° ',
    'Exp': 4.87}},
  '24881579': {'PMID': '24881579',
   'Title': 'Transcriptome and exoproteome analysis of utilization of plant-derived biomass by Myceliophthora thermophila.',
   'mt002001': {'Sample ID': 'mt002001',
    'Condition': 'wt Oat 45° early growth stage',
    'Exp': 4.78},
   'mt002002': {'Sample ID': 'mt002002'

In [11]:
#将output_dict 转为json文件
with open('gene_exp.json', 'w') as f:
    json.dump(dict_gene_exp, f, indent=4)